In [1]:
import os
import torch

from tqdm import tqdm
from pathlib import Path
from rich.table import Table
from rich.console import Console

from utils import load_shards, save_run, seed_everything

 
SEED = 7
SCRIPT_PATH = "classification/mlp_cls_ablation.py"
BACKBONE = os.environ.get("BACKBONE", "dinov3-vit7b16-pretrain-lvd1689m")
IMAGE_SIZE = os.environ.get("IMAGE_SIZE", 448)
BACKGROUND_AUG = "normal"
FINAL_LAYER_CLASSIFIER_METHOD = "mlp"
EXPERIMENT_ID = f"{BACKBONE}_{BACKGROUND_AUG}_{IMAGE_SIZE}_{FINAL_LAYER_CLASSIFIER_METHOD}"

MODEL_TRAIN = Path(f"facebook/{BACKBONE}/bfloat16_normal_{IMAGE_SIZE}/train")
MODEL_VAL = Path(f"facebook/{BACKBONE}/bfloat16_normal_{IMAGE_SIZE}/val")
MODEL_TEST = Path(f"facebook/{BACKBONE}/bfloat16_normal_{IMAGE_SIZE}/test")
ROOT = Path("/home/cavadalab/Documents/scsv/fungitastic2026_2/data_processed")

BATCH_SIZE = 64
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 200
PATIENCE = 30
MIN_DELTA = 1e-4
VAL_FRACTION = 0.15
PROJECTION_DIM = 256
HIDDEN_DIM = 512
NUM_REGISTER_TOKENS = 4


In [3]:
train_data = load_shards(ROOT / MODEL_TRAIN)
val_data = load_shards(ROOT / MODEL_VAL)
test_data = load_shards(ROOT / MODEL_TEST)

Loading test: 100%|██████████| 20/20 [00:00<00:00, 59.32shard/s]


In [4]:
train_data

{'labels': tensor([2359, 2359, 2359,  ..., 1733, 2394, 2394]),
 'cls_tokens': tensor([[ 0.2119, -0.1494,  0.0845,  ...,  0.0176,  0.0228, -0.2520],
         [ 0.3262,  0.0299,  0.0767,  ..., -0.0923,  0.0244, -0.1235],
         [ 0.0006, -0.2344,  0.1377,  ...,  0.0118,  0.0674, -0.1416],
         ...,
         [-0.0664, -0.1562,  0.1670,  ..., -0.0874, -0.1040,  0.0078],
         [ 0.0422, -0.2158, -0.0322,  ...,  0.0684,  0.0168, -0.1494],
         [ 0.0505, -0.2344, -0.0869,  ...,  0.1484, -0.0334, -0.2021]]),
 'register_tokens': tensor([[[ 2.6172e-01, -2.0312e-01,  5.4932e-02,  ...,  4.5166e-03,
            3.4668e-02, -2.3633e-01],
          [ 7.9346e-03,  2.1362e-02,  2.2583e-02,  ..., -1.5503e-02,
            7.7515e-03, -2.6611e-02],
          [ 9.3750e-02, -3.5889e-02, -2.6245e-02,  ..., -4.3164e-01,
            9.3262e-02, -3.6377e-02],
          [ 2.9907e-02,  1.7578e-01,  2.8198e-02,  ...,  2.1191e-01,
            6.1035e-02, -4.1260e-02]],
 
         [[ 3.7500e-01, -8.0566

In [7]:
train_data['mean_pooled_gt_masked_patch_tokens'].shape

torch.Size([13777, 4096])

In [8]:
train_data['mean_pooled_sam_masked_patch_tokens'].shape

torch.Size([13777, 4096])